# AAPL Multi-Model Options Backtest

This notebook trains:
1. a long-history AAPL trade selector on price plus ratio/metrics features,
2. a pairwise learn-to-rank option selector on per-chain contract features,
3. a hybrid mean-variance option selector trained on quant-warehouse `hybrid` labels, which blend return and rank signals.

It then backtests 2025+ and compares the fixed buckets against both ML option selectors.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

repo_root = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'quant_orchestrator').is_dir()
)
warehouse_root = repo_root.parent / 'quant-warehouse'
if not (warehouse_root / 'quant_warehouse').is_dir():
    raise RuntimeError(f'Could not find quant-warehouse next to {repo_root}')

for path in (repo_root, warehouse_root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from quant_orchestrator.options_research import (
    backtest_trade_variants,
    build_daily_feature_frame,
    build_option_training_panel,
    build_trade_candidates,
    build_trade_feature_panel,
    load_fundamental_panel,
    load_price_history,
    maybe_backfill_aapl_options,
    score_trade_candidates,
    select_best_trade_per_date,
    summarize_variant_curves,
    train_pairwise_option_ranker,
    train_trade_selector_model,
)

ARTIFACT_DIR = repo_root / 'artifacts' / 'aapl_options_multi_model_backtest'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo_root={repo_root}')
print(f'warehouse_root={warehouse_root}')
print(f'artifact_dir={ARTIFACT_DIR}')

In [ ]:
SYMBOL = 'AAPL'
TRAIN_CUTOFF = pd.Timestamp('2025-01-01')
TRADE_START = '2018-01-01'
TRADE_END = None
RUN_BACKFILL = False
BACKFILL_START = '2024-01-01'
BACKFILL_END = pd.Timestamp.today().normalize().date().isoformat()
DOWNLOAD_MISSING = False
MAX_DTE = 90
STRIKE_RANGE = 12
TARGET_TENOR_DAYS = 60

if RUN_BACKFILL:
    backfill_summary = maybe_backfill_aapl_options(
        start_date=BACKFILL_START,
        end_date=BACKFILL_END,
        symbol=SYMBOL,
        overwrite=False,
        skip_existing=True,
    )
    print(json.dumps(backfill_summary, indent=2, default=str))

In [ ]:
prices = load_price_history(SYMBOL, start=TRADE_START, end=TRADE_END)
fundamentals = load_fundamental_panel(SYMBOL, sections=('ratios', 'metrics'), start=TRADE_START, end=TRADE_END)
daily_features = build_daily_feature_frame(prices, fundamentals)
trades = build_trade_candidates(prices, start=TRADE_START, end=TRADE_END)
trade_panel = build_trade_feature_panel(trades, daily_features)

print('prices', prices.shape, prices.index.min(), prices.index.max())
print('fundamentals', fundamentals.shape, fundamentals.index.min(), fundamentals.index.max())
print('daily_features', daily_features.shape)
print('trades', trades.shape)
print('trade_panel', trade_panel.shape)

display(trades.head(10))

In [ ]:
trade_train = trade_panel.loc[trade_panel['entry_date'] < TRAIN_CUTOFF].copy()
trade_eval = trade_panel.loc[trade_panel['entry_date'] >= TRAIN_CUTOFF].copy()

trade_model = train_trade_selector_model(trade_train, target_col='target_return', model_kind='regressor')
trade_panel = trade_panel.copy()
trade_panel['trade_selector_score'] = score_trade_candidates(trade_model, trade_panel)
selected_trades = select_best_trade_per_date(trade_panel.assign(score=trade_panel['trade_selector_score']))
selected_train = selected_trades.loc[selected_trades['entry_date'] < TRAIN_CUTOFF].copy()
selected_eval = selected_trades.loc[selected_trades['entry_date'] >= TRAIN_CUTOFF].copy()

print(trade_model.metrics)
print('selected_trades', selected_trades.shape)
print('selected_train', selected_train.shape)
print('selected_eval', selected_eval.shape)

display(selected_trades.head(10))

In [ ]:
option_train_panel = build_option_training_panel(
    selected_train,
    download_missing=DOWNLOAD_MISSING,
    max_dte=MAX_DTE,
    strike_range=STRIKE_RANGE,
)

if option_train_panel.empty:
    raise RuntimeError('No option training panel could be built from pre-2025 selected trades')

rank_train_panel = option_train_panel.loc[option_train_panel['label_method'].eq('rank')].copy()
hybrid_train_panel = option_train_panel.loc[option_train_panel['label_method'].eq('hybrid')].copy()
mean_var_train_panel = option_train_panel.loc[option_train_panel['label_method'].eq('mean_variance')].copy()

option_ranker = train_pairwise_option_ranker(rank_train_panel)
hybrid_option_model = train_trade_selector_model(
    hybrid_train_panel,
    target_col='target_value',
    model_kind='regressor',
    extra_exclude=(
        'contract_symbol',
        'trade_id',
        'trade_entry_date',
        'trade_exit_date',
        'entry_snapshot_date',
        'exit_snapshot_date',
        'option_return_pct',
        'rank_y',
        'mv_weight',
        'label',
        'target_value',
        'target_col',
        'label_method',
        'task_name',
    ),
)

print('option_train_panel', option_train_panel.shape)
print('rank_train_panel', rank_train_panel.shape)
print('hybrid_train_panel', hybrid_train_panel.shape)
print('mean_var_train_panel', mean_var_train_panel.shape)
print('option_ranker', option_ranker.metrics)
print('hybrid_option_model', hybrid_option_model.metrics)

display(option_train_panel.head(10))

In [ ]:
option_eval_panel = build_option_training_panel(
    selected_eval,
    download_missing=DOWNLOAD_MISSING,
    max_dte=MAX_DTE,
    strike_range=STRIKE_RANGE,
)

if option_eval_panel.empty:
    raise RuntimeError('No option evaluation panel could be built for 2025+ selected trades')

variant_summary, variant_curves = backtest_trade_variants(
    selected_eval,
    option_panel=option_eval_panel,
    option_ranker=option_ranker,
    mv_option_model=hybrid_option_model,
    target_tenor_days=TARGET_TENOR_DAYS,
)
curve_summary = summarize_variant_curves(variant_curves)

print('option_eval_panel', option_eval_panel.shape)
display(variant_summary.head(10))
display(curve_summary)

In [ ]:
joblib.dump(trade_model, ARTIFACT_DIR / 'aapl_trade_selector.joblib')
joblib.dump(option_ranker, ARTIFACT_DIR / 'aapl_option_ranker.joblib')
joblib.dump(hybrid_option_model, ARTIFACT_DIR / 'aapl_hybrid_option_model.joblib')
selected_trades.to_parquet(ARTIFACT_DIR / 'selected_trades.parquet', index=False)
selected_eval.to_parquet(ARTIFACT_DIR / 'selected_eval_trades_2025_plus.parquet', index=False)
option_train_panel.to_parquet(ARTIFACT_DIR / 'option_train_panel.parquet', index=False)
option_eval_panel.to_parquet(ARTIFACT_DIR / 'option_eval_panel.parquet', index=False)
variant_summary.to_parquet(ARTIFACT_DIR / 'variant_trade_summary.parquet', index=False)
curve_summary.to_csv(ARTIFACT_DIR / 'variant_curve_summary.csv', index=False)

with open(ARTIFACT_DIR / 'run_metadata.json', 'w') as f:
    json.dump(
        {
            'symbol': SYMBOL,
            'train_cutoff': str(TRAIN_CUTOFF.date()),
            'trade_start': TRADE_START,
            'trade_end': TRADE_END,
            'max_dte': MAX_DTE,
            'strike_range': STRIKE_RANGE,
            'target_tenor_days': TARGET_TENOR_DAYS,
            'download_missing': DOWNLOAD_MISSING,
            'option_train_rows': int(len(option_train_panel)),
            'rank_rows': int(len(rank_train_panel)),
            'hybrid_rows': int(len(hybrid_train_panel)),
        },
        f,
        indent=2,
        default=str,
    )

print('saved artifacts to', ARTIFACT_DIR)